### 1. Plot diurnal cycles from model data
Load series extracted from blurred data
Groups data by season, then averages by hour of day to give the diurnal cycle.

In [10]:
from Montreal_UHI_toolbox import *
import plotly.express as px
import plotly.graph_objects as go
station_set = obs

In [ ]:
series = {}
for key in ['tas_C','tas_T']:
    series[key] = xr.open_mfdataset(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}.nc')
    series[key]

In [75]:
diurnal = {}
for key in ['tas_C','tas_T']:
    diurnal[key] = {}
    for season in ['JJA','SON','DJF','MAM']:
        diurnal[key][season] = series[key].sel(time=series[key].time.dt.season == season).groupby('time.hour').mean(dim='time')

In [ ]:
# Display temperatures

for season in ['JJA','SON','DJF','MAM']:
    fig = go.Figure()
    for key,model in zip(['tas_C','tas_T'],['CLASS','TEB+CLASS']):
                     
        for point,name in zip(diurnal[key][season].points.values, station_set.station_name.values): # assuming point was indexed by station_names
            fig.add_trace(go.Scatter(x=diurnal[key][season].hour.values,y=diurnal[key][season].sel(points=point).tas.values - 273.15,name=f'{name} ({model})'))
    
    fig.update_layout(title=f'{season} Diurnal Cycle',)
    
    fig.update_layout(
        title=dict(text=f'{season} Diurnal Cycle'),
        xaxis=dict(title=dict(text='hour of day')),
        yaxis=dict(title=dict(text='temperature (°C)')),
        legend=dict(title=dict(text='station and surface scheme')),
    )
    fig.show()
    # fig.write_html(f'/runoff/gulley/plots/prelim_{season}_diurnal_simobs.html')

### 2. Plot annual cycles from model and observational data
Groups and averages model & station data by month

In [ ]:
# Load all daily/subdaily simobs data
series = {}
for key in ['tas_C','tas_T','tasmin_C','tasmin_T','tasmax_C','tasmax_T']:
    series[key] = xr.open_mfdataset(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}.nc')
    series[key]

# Load all daily obs data
# [observation data loading into series[tas_S etc] goes here]

# Calculate monthly averages of simobs
monthly = {}
for key in ['tas_C','tas_T','tasmin_C','tasmin_T','tasmax_C','tasmax_T']: 
    monthly[key] = series[key].groupby('time.month').mean(dim='time')

# Calculate monthly averages of obs
# [monthly averages of obs here]

In [ ]:
for f,field_name in zip(['tasmin','tasmax'],['minimum daily temperature','maximum daily temperature']):
    fig = go.Figure()
    for key,model in zip([f'{f}_C',f'{f}_T'],['CLASS','TEB+CLASS']):
        for point,name in zip(monthly[key].points.values, station_set.station_name.values):
            fig.add_trace(go.Scatter(x=monthly[key].sel(points=point).month.values,y=monthly[key].sel(points=point)[f].values - 273.15,name=f'{name} ({model})'))

    fig.update_layout(
        title=dict(text=f'Monthly averages of {field_name}'),
        xaxis=dict(title=dict(text='month of year')),
        yaxis=dict(title=dict(text='temperature (°C)')),
        legend=dict(title=dict(text='station and surface scheme')),
    )
    # fig.show()
    # fig.write_html(f'/runoff/gulley/plots/prelim_{f}_annual_cycle_simobs.html')

In [ ]:
fig = go.Figure()

for point,name in zip(monthly[key].points.values, station_set.station_name.values):
    fig.add_trace(go.Scatter(x=monthly['tasmin_C'].sel(points=point).month.values,y=monthly['tasmax_C'].sel(points=point)['tasmax'].values - monthly['tasmin_C'].sel(points=point)['tasmin'].values,name=f'{name} (CLASS)'))
    fig.add_trace(go.Scatter(x=monthly['tasmin_T'].sel(points=point).month.values,y=monthly['tasmax_T'].sel(points=point)['tasmax'].values - monthly['tasmin_T'].sel(points=point)['tasmin'].values,name=f'{name} (TEB+CLASS)'))

fig.update_layout(
    title=dict(text=f'Monthly averages of temperature spread'),
    xaxis=dict(title=dict(text='month of year')),
    yaxis=dict(title=dict(text='T<sub>max</sub> - T<sub>min</sub> (°C)')),
    legend=dict(title=dict(text='station and surface scheme')),
)
# fig.show()
# fig.write_html(f'/runoff/gulley/plots/prelim_spread_annual_cycle_simobs.html')